# UPDATE METADATA
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [4]:
level = '2'

In [2]:
file_codes = ['UCSDVOL']
#'ADNIMERGE', 'UCSFFSX', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL'] #, 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 
# 'BLCHANGE', 'DXSUM', 

In [5]:
search = client.query_files(
    query={'custom.level' : 'cleaned_0'+level, 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [6]:
print(zip_files.keys())

dict_keys(['UCSDVOL_11Aug2025_02.csv'])


# Import support file already populated

In [7]:
support_file = pd.read_excel('ADNI_variables_cleaned'+ level +'.xlsx')
dataCleaner = DataCleaner(support_file=support_file)

# Get & update metadata

In [8]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    final_df = df.copy(deep=True)
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_0'+level, file_name=file_name, prefix='cleaned/single_file', updated_support_file=support_file)  

    metadata = client.get_metadata(object_name = 'cleaned/single_file/' + file_name)
    metadata_id = metadata['metadata']['_id']

    result = client.update_file(
        object_name= 'cleaned/single_file/' + file_name,
        metadata=updated_metadata
    )

    print('\n###  ', file_name, '  ##################\n', updated_metadata)


###   UCSDVOL_11Aug2025_02.csv   ##################
 {'cofattori': [], 'file_code': 'UCSDVOL', 'level': 'cleaned_02', 'norm_intervallo': [], 'norm_scala': [], 'norm_scale_value': [], 'norm_volume': ['Brain', 'ICV', 'Ventricles', 'LHippocampus', 'RHippocampus'], 'population': ['ADNI1'], 'predittori': ['Brain', 'ICV', 'Ventricles', 'LHippocampus', 'RHippocampus'], 'source': 'ADNI', 'volume_norm_values': {'Ventricles': [9942.24, 116400.08, 'increasing'], 'ICV': [1178000.0, 1950280.0, 'inverse'], 'LHippocampus': [1866.23, 4715.54, 'inverse'], 'Brain': [754149.98, 1244134.8, 'increasing']}}


## Verify uploaded metadata

In [17]:
metadata = client.get_metadata(object_name='cleaned/single_file/UCSFFSX7_11Aug2025_02.csv')
print(metadata['metadata']['custom'])

{'cofattori': [], 'file_code': 'UCSFFSX7', 'level': 'cleaned_02', 'norm_intervallo': [], 'norm_scala': [], 'norm_scale_value': [], 'norm_volume': ['LVentricle', 'ICV', 'LEntorhinal', 'LFusiform', 'LHippocampus', 'LMidTemp', 'RVentricle', 'REntorhinal', 'RFusiform', 'RHippocampus', 'RMidTemp'], 'population': ['ADNI4'], 'predittori': ['LVentricle', 'ICV', 'LEntorhinal', 'LFusiform', 'LHippocampus', 'LMidTemp', 'RVentricle', 'REntorhinal', 'RFusiform', 'RHippocampus', 'RMidTemp'], 'source': 'ADNI', 'volume_norm_values': {'ICV': [1178000.0, 1950280.0, 'inverse'], 'LEntorhinal': [695.0, 2900.02, 'inverse'], 'LFusiform': [5114.03, 12232.97, 'inverse'], 'LHippocampus': [1866.23, 4715.54, 'inverse'], 'LMidTemp': [5323.8, 13798.0, 'inverse']}}
